[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Layers and Models &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import contextlib
import os
import warnings

import numpy as np
import torch
from torch import nn


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "| keras", keras.__version__,
      "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | keras 3.13.2 | 1200 stations


**1.** Build an `nn.Sequential` with two hidden layers of 16 units, each followed by `nn.ReLU()`, and one output, then print it and its number of parameters.

In [2]:
torch.manual_seed(0)
two_hidden = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 16), nn.ReLU(), nn.Linear(16, 1))
print(two_hidden)
print(sum(p.numel() for p in two_hidden.parameters()), "parameters")

Sequential(
  (0): Linear(in_features=4, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=16, bias=True)
  (3): ReLU()
  (4): Linear(in_features=16, out_features=1, bias=True)
)
369 parameters


The first hidden layer holds 4 × 16 weights and 16 biases, 80 in all, the second 16 × 16 and 16, 272, and the output 16 and 1, 17, which make 369.

**2.** Write a subclass of `nn.Module`, `TwoLayerNet`, whose `__init__` takes the sizes of two hidden layers, and print the name and shape of every parameter for hidden sizes of 8 and 4.

In [3]:
class TwoLayerNet(nn.Module):
    def __init__(self, first, second):
        super().__init__()
        self.first = nn.Linear(4, first)
        self.second = nn.Linear(first, second)
        self.output = nn.Linear(second, 1)

    def forward(self, x):
        return self.output(torch.relu(self.second(torch.relu(self.first(x)))))


for name, parameter in TwoLayerNet(8, 4).named_parameters():
    print(f"{name:<14} {tuple(parameter.shape)}")

first.weight   (8, 4)
first.bias     (8,)
second.weight  (4, 8)
second.bias    (4,)
output.weight  (1, 4)
output.bias    (1,)


Each weight is outputs by inputs: 8 by 4, then 4 by 8, then 1 by 4.

**3.** Build the network from task 1 with Keras's functional API, with named layers, and check that `count_params()` matches the PyTorch count.

In [4]:
keras.utils.set_random_seed(0)
inputs = keras.Input((4,), name="features")
x = keras.layers.Dense(16, activation="relu", name="first")(inputs)
x = keras.layers.Dense(16, activation="relu", name="second")(x)
outputs = keras.layers.Dense(1, name="output")(x)
two_hidden_keras = keras.Model(inputs, outputs, name="two_hidden")

torch_count = sum(p.numel() for p in nn.Sequential(
    nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 16), nn.ReLU(), nn.Linear(16, 1)).parameters())
print(two_hidden_keras.count_params(), torch_count)

369 369


Both count 369 parameters.

**4.** Copy the weights of the PyTorch network from task 1, made from `torch.manual_seed(0)`, into the Keras network from task 3, and print both models' predictions for the first three test stations.

In [5]:
torch.manual_seed(0)
two_hidden = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 16), nn.ReLU(), nn.Linear(16, 1))

keras.utils.set_random_seed(0)
inputs = keras.Input((4,), name="features")
x = keras.layers.Dense(16, activation="relu", name="first")(inputs)
x = keras.layers.Dense(16, activation="relu", name="second")(x)
outputs = keras.layers.Dense(1, name="output")(x)
two_hidden_keras = keras.Model(inputs, outputs, name="two_hidden")

for torch_layer, name in zip((two_hidden[0], two_hidden[2], two_hidden[4]), ("first", "second", "output")):
    two_hidden_keras.get_layer(name).set_weights(
        [torch_layer.weight.detach().numpy().T, torch_layer.bias.detach().numpy()])

stations = scaled[test_rows[:3]]
with torch.no_grad():
    print("PyTorch:", two_hidden(torch.from_numpy(stations)).numpy().ravel().round(4))
print("Keras:  ", two_hidden_keras.predict(stations, verbose=0).ravel().round(4))

PyTorch: [-0.2236 -0.2193 -0.1485]
Keras:   [-0.2236 -0.2193 -0.1485]


The predictions agree to four decimal places. The three linear layers sit at positions 0, 2 and 4 of the `nn.Sequential`, because the `nn.ReLU()` layers between them hold no weights.

**5.** Train a `StationNet` made from `torch.manual_seed(0)` with `torch.optim.Adam(model.parameters(), lr=0.01)` and `nn.MSELoss()` for 500 full-batch steps on the training stations, and print its loss on the test stations.

In [6]:
class StationNet(nn.Module):
    """Four features, one hidden layer of `hidden` units, and one output."""

    def __init__(self, hidden=32):
        super().__init__()
        self.hidden = nn.Linear(4, hidden)
        self.output = nn.Linear(hidden, 1)

    def forward(self, x):
        return self.output(torch.relu(self.hidden(x)))


X_train = torch.from_numpy(scaled[train_rows])
y_train = torch.from_numpy(jan_mean[train_rows]).unsqueeze(1)
X_test = torch.from_numpy(scaled[test_rows])
y_test = torch.from_numpy(jan_mean[test_rows]).unsqueeze(1)

torch.manual_seed(0)
model = StationNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()
for step in range(500):
    optimizer.zero_grad()
    loss = loss_fn(model(X_train), y_train)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    print("training loss", round(loss.item(), 3), "| test loss", round(loss_fn(model(X_test), y_test).item(), 3))

training loss 0.782 | test loss 0.96


The training loss is 0.782 and the test loss 0.96, the same as the `nn.Sequential` network in the **Why PyTorch and TensorFlow** notebook, because the same two layers are created in the same order from the same seed.

---

&#8592; **Back to:** [Layers and Models](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/06-layers-and-models.ipynb)  &nbsp;&middot;&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)
